In [ ]:

def get_text_emb(model):
    import os 
    import json
    import pandas as pd
    import torch


    #json 데이터 파일 경로 반환 함수: get_files(경로명)
    def get_files(path):
        json_files = []
        for root, dirs, files in os.walk(path):
            for file in files:
                if file.endswith('.json'):
                    json_files.append(os.path.join(root, file))
        return json_files
    
    #입력된 json 데이터를 데이터프레임으로 만드는 함수: make_dataframe(파일명)
    def make_dataframe(data):
        info = data['info']
        list_data = data['list']
        rows = []
        for i, item in enumerate(list_data, start=1):
            text = [] 
            for sub_item in item['list']:
                if 'audio' in sub_item:
                    for audio in sub_item['audio']:
                        if audio['type'] == "Q":
                            speaker = "상담자"
                        else: 
                            speaker = "내담자"
                        text.append(f"{speaker}: {audio['text']}")
            문항대화 =' '.join(text)
            rows.append({
                "id": f"{info['ID']}-{i}",
                "대화": 문항대화
            })
        return pd.DataFrame(rows)
    
    #train 데이터 데이터프레임화하기 
    json_files = get_files('Data/Training/02/')
    df_list = [] 
    for item in json_files: 
        with open(item, 'r', encoding='utf-8') as file:
            data = json.load(file)
        df = make_dataframe(data)
        df_list.append(df)
    dfs = pd.concat(df_list, ignore_index = True)
    dfs.sort_values(by="id") # 마지막에 합치기(id 오름차순)
    text_train = dfs['대화'].tolist() #임베딩할 부분만 리스트형으로 추출(리스트로 받음)
    
    #validation 데이터 데이터프레임화하기
    json_files = get_files('Data/Validation/02/')
    df_list = [] 
    for item in json_files: 
        with open(item, 'r', encoding='utf-8') as file:
            data = json.load(file)
        df = make_dataframe(data)
        df_list.append(df)
    dfs = pd.concat(df_list, ignore_index = True)
    dfs.sort_values(by="id") 
    text_valid = dfs['대화'].tolist() 

    # 배치별 임베딩하는 함수: embed_texts(임베딩할 텍스트)
    def embed_texts(texts):
        embeddings = []
        batch_size=512
        
        device = "cuda:0"
        model.to(device)
        model.eval()

        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]

            # 입력테스트 토큰화 및 텐서변환
            inputs = tokenizer(batch_texts, padding="longest", truncation=True, return_tensors="pt", max_length=512)
            # GPU로 텐서이동
            input_ids = inputs["input_ids"].to(device)
            attention_mask = inputs["attention_mask"].to(device) #패딩된 부분 무시하도록 함
            # 임베딩
            with torch.no_grad():
                outputs = model(input_ids, attention_mask=attention_mask)
                batch_embeddings = outputs.last_hidden_state[:, 0, :]  # 마지막 출력값의 임베딩
                embeddings.append(batch_embeddings.cpu())  # GPU -> CPU로 이동
            
            print(f"텍스트 임베딩 중: batch {i},{i/len(texts)*100:.0f}%")

        embedded = torch.cat(embeddings, dim=0) # 배치별 임베딩을 하나의 텐서로 합침
        print("임베딩 완료. Embeddings shape:", embedded.shape)
        return embedded
    
    if model == "KoELECTRA":
        from transformers import ElectraTokenizer, ElectraModel
        MODEL_NAME = "monologg/koelectra-base-v3-discriminator"
        tokenizer = ElectraTokenizer.from_pretrained(MODEL_NAME)
        model = ElectraModel.from_pretrained(MODEL_NAME)
    elif model == "KoBERT":
        from transformers import BertTokenizer, BertModel
        MODEL_NAME = "monologg/kobert"
        tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
        model = BertModel.from_pretrained(MODEL_NAME)
    elif model == "KcELECTRA":
        from transformers import ElectraTokenizer, ElectraModel
        MODEL_NAME = "beomi/KcELECTRA-base-v2022"
        tokenizer = ElectraTokenizer.from_pretrained(MODEL_NAME)
        model = ElectraModel.from_pretrained(MODEL_NAME)
    elif model == "KoRoBERTa":
        from transformers import AutoTokenizer, AutoModel
        MODEL_NAME = "klue/roberta-base"
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        model = AutoModel.from_pretrained(MODEL_NAME)

     

    else:
        return "모델이 없습니다."

    text_embed_train = embed_texts(text_train)
    text_embed_valid = embed_texts(text_valid)

    return text_embed_train, text_embed_valid



In [3]:
text_embed_train, text_embed_valid = get_text_emb("KoRoBERTa")
text_embed_valid[0]
text_embed_valid[0]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

2024-11-24 16:24:52.359417: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1732433092.378561    8062 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1732433092.384574    8062 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-24 16:24:52.403865: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at klue/roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


텍스트 임베딩 중: batch 0,0%
텍스트 임베딩 중: batch 512,3%
텍스트 임베딩 중: batch 1024,5%
텍스트 임베딩 중: batch 1536,8%
텍스트 임베딩 중: batch 2048,10%
텍스트 임베딩 중: batch 2560,13%
텍스트 임베딩 중: batch 3072,15%
텍스트 임베딩 중: batch 3584,18%
텍스트 임베딩 중: batch 4096,20%
텍스트 임베딩 중: batch 4608,23%
텍스트 임베딩 중: batch 5120,25%
텍스트 임베딩 중: batch 5632,28%
텍스트 임베딩 중: batch 6144,31%
텍스트 임베딩 중: batch 6656,33%
텍스트 임베딩 중: batch 7168,36%
텍스트 임베딩 중: batch 7680,38%
텍스트 임베딩 중: batch 8192,41%
텍스트 임베딩 중: batch 8704,43%
텍스트 임베딩 중: batch 9216,46%
텍스트 임베딩 중: batch 9728,48%
텍스트 임베딩 중: batch 10240,51%
텍스트 임베딩 중: batch 10752,53%
텍스트 임베딩 중: batch 11264,56%
텍스트 임베딩 중: batch 11776,58%
텍스트 임베딩 중: batch 12288,61%
텍스트 임베딩 중: batch 12800,64%
텍스트 임베딩 중: batch 13312,66%
텍스트 임베딩 중: batch 13824,69%
텍스트 임베딩 중: batch 14336,71%
텍스트 임베딩 중: batch 14848,74%
텍스트 임베딩 중: batch 15360,76%
텍스트 임베딩 중: batch 15872,79%
텍스트 임베딩 중: batch 16384,81%
텍스트 임베딩 중: batch 16896,84%
텍스트 임베딩 중: batch 17408,86%
텍스트 임베딩 중: batch 17920,89%
텍스트 임베딩 중: batch 18432,92%
텍스트 임베딩 중: batch 18944,94%
텍스

tensor([ 1.6289e-01, -7.3311e-01, -2.0069e-01, -3.7041e-02,  3.9096e-02,
        -3.5751e-01, -1.1790e-01, -2.3026e-01, -3.1149e-01, -8.4284e-02,
        -2.7104e-01,  3.6658e-02, -3.5635e-01,  1.8516e-02, -7.0849e-02,
        -5.8863e-02, -5.4099e-03, -3.6851e-01, -1.8951e-01, -3.0962e-01,
        -7.5068e-02, -1.6417e-01,  1.9567e-01,  2.5418e-01, -8.5991e-02,
        -1.4023e-01,  8.8862e-02, -2.8673e-01,  5.4760e-02,  1.7657e-02,
        -1.2095e-02, -6.0084e-01, -9.5266e-02, -2.5921e-01,  8.9772e-01,
        -2.9106e-01, -7.2683e-02, -1.0541e-01, -5.3930e-02,  4.6050e-02,
         1.7160e-01,  4.2833e-02,  1.7461e-01,  1.0027e-01,  3.3550e-01,
        -7.3871e-02, -2.3124e-01, -8.8156e-02, -7.3275e-01, -3.2828e-01,
         3.6923e-01, -6.7928e-02,  1.7959e-01, -1.9408e-01,  7.8268e-02,
         8.7135e-02, -1.6910e-01,  1.7743e-02, -6.7902e-02,  8.4341e-02,
         1.3399e-01,  2.7168e-01, -2.3285e-01,  3.1711e-01,  1.3389e-01,
         1.2900e-01, -2.1706e-01, -2.5825e-01, -1.3

In [18]:
!nvidia-smi

Sun Nov 24 15:54:59 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 530.30.02              Driver Version: 530.30.02    CUDA Version: 12.1     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                  Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf            Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3090         Off| 00000000:65:00.0 Off |                  N/A |
| 50%   28C    P8               22W / 350W|   3670MiB / 24576MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [4]:
import os
import torch

# 저장할 디렉토리 경로
save_dir = 'Embeddings_training/text/'
filename = 'text_embeddings_KoRoBERTa.pt'
# 전체 경로 생성
full_path = os.path.join(save_dir, filename)
# 임베딩 저장
torch.save(text_embed_train, full_path)

# 테스트셋
save_dir = 'Embeddings_test/text/'
filename = 'text_embeddings_KoRoBERTa.pt'
full_path = os.path.join(save_dir, filename)
torch.save(text_embed_valid, full_path)